# 20 Seq2Seq与注意力机制 - Transformer的前身

本教程将学习**序列到序列(Seq2Seq)架构**和**注意力(Attention)机制**，这些是Transformer的核心思想来源。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math

torch.manual_seed(42)
np.random.seed(42)

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("导入库成功！")

## 一、Seq2Seq（序列到序列）架构

### 为什么需要Seq2Seq？

之前的模型都是"固定长度输入→固定长度输出"。但很多任务需要**变长序列→变长序列**：

| 任务 | 输入 | 输出 |
|------|------|------|
| 机器翻译 | "你好世界" (4字符) | "Hello World" (10字符) |
| 文本摘要 | 长文章 | 短摘要 |
| 对话系统 | 用户问题 | 机器回复 |
| 语音识别 | 语音帧序列 | 文字序列 |

### 核心思想：编码器-解码器

**编码器（Encoder）**：
- 读取整个输入序列
- 压缩成一个**上下文向量（Context Vector）**
- 这个向量应该包含输入的全部语义信息

**解码器（Decoder）**：
- 从上下文向量开始
- 逐步生成输出序列
- 每一步产生一个token，并将其作为下一步的输入

```
编码器                   解码器
┌───┐   ┌───┐           ┌───┐   ┌───┐   ┌───┐
│你好│→│世界│→ [Context] →│<SOS>│→│Hello│→│World│→...
└───┘   └───┘           └───┘   └───┘   └───┘
  RNN     RNN             LSTM     LSTM    LSTM
```

In [ ]:
# 构建简单的Seq2Seq模型
print("=" * 70)
print("Seq2Seq模型构建")
print("=" * 70)
print()

class Encoder(nn.Module):
    def __init__(self, input_size, hidden_size, n_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.rnn = nn.GRU(hidden_size, hidden_size, n_layers, batch_first=True)
    
    def forward(self, src):
        # src shape: (batch, seq_len)
        embedded = self.embedding(src)  # (batch, seq_len, hidden)
        outputs, hidden = self.rnn(embedded)
        return outputs, hidden

class Decoder(nn.Module):
    def __init__(self, output_size, hidden_size, n_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.rnn = nn.GRU(hidden_size, hidden_size, n_layers, batch_first=True)
        self.fc_out = nn.Linear(hidden_size, output_size)
    
    def forward(self, input_token, hidden):
        # input_token: (batch, 1)
        embedded = self.embedding(input_token)  # (batch, 1, hidden)
        output, hidden = self.rnn(embedded, hidden)
        prediction = self.fc_out(output.squeeze(1))  # (batch, output_size)
        return prediction, hidden

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, max_len):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.max_len = max_len
    
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        output_size = self.decoder.fc_out.out_features
        
        outputs = torch.zeros(batch_size, trg_len, output_size)
        
        # 编码
        enc_outputs, hidden = self.encoder(src)
        
        # 解码：第一个输入是<SOS>
        decoder_input = trg[:, 0].unsqueeze(1)
        
        for t in range(1, trg_len):
            output, hidden = self.decoder(decoder_input, hidden)
            outputs[:, t] = output
            
            # Teacher Forcing: 有时用真实标签，有时用自己的预测
            top1 = output.argmax(1)
            use_teacher_forcing = torch.rand(1).item() < teacher_forcing_ratio
            if use_teacher_forcing:
                decoder_input = trg[:, t].unsqueeze(1)
            else:
                decoder_input = top1.unsqueeze(1)
        
        return outputs

# 创建模型
input_vocab_size = 100  # 输入词表大小
output_vocab_size = 100 # 输出词表大小
hidden_size = 32
max_len = 15

encoder = Encoder(input_vocab_size, hidden_size)
decoder = Decoder(output_vocab_size, hidden_size)
model = Seq2Seq(encoder, decoder, max_len)

print(f"模型结构:")
print(model)
print(f"\n总参数量: {sum(p.numel() for p in model.parameters()):,}")
print(f"Encoder参数量: {sum(p.numel() for p in encoder.parameters()):,}")
print(f"Decoder参数量: {sum(p.numel() for p in decoder.parameters()):,}")

In [ ]:
# Seq2Seq训练演示（小规模数据）
print("\n=== Seq2Seq训练演示 ===")
print()

torch.manual_seed(42)

# 创建模拟数据：学习"反转序列"的任务
# 输入: [a, b, c, d] → 输出: [d, c, b, a]
def generate_reverse_data(n_samples, seq_len, vocab_size):
    X = torch.randint(2, vocab_size, (n_samples, seq_len))
    # 加入SOS/EOS标记
    SOS = torch.zeros(n_samples, 1, dtype=torch.long)
    y = torch.cat([SOS, torch.flip(X, dims=[1])], dim=1)
    return X, y

vocab_size = 20  # 词汇表（0=SOS/EOS, 1=PAD, 2~19=实际词）
seq_len = 6

X_train, y_train = generate_reverse_data(500, seq_len, vocab_size)
X_test, y_test = generate_reverse_data(100, seq_len, vocab_size)

print(f"训练数据: X={X_train.shape}, y={y_train.shape}")
print(f"输入示例: {X_train[0].tolist()}")
print(f"目标示例: {y_train[0].tolist()}")
print(f"(第一个0是SOS, 后面是反转后的序列)")
print()

# 训练
torch.manual_seed(42)
enc = Encoder(vocab_size, hidden_size=32)
dec = Decoder(vocab_size, hidden_size=32)
model = Seq2Seq(enc, dec, max_len=seq_len+1)

optimizer = optim.Adam(model.parameters(), lr=0.005)
criterion = nn.CrossEntropyLoss(ignore_index=1)  # 忽略PAD

losses = []
accuracies = []

for epoch in range(100):
    model.train()
    optimizer.zero_grad()
    output = model(X_train, y_train, teacher_forcing_ratio=0.7)
    
    # 计算损失
    loss = criterion(output[:, 1:].reshape(-1, vocab_size), y_train[:, 1:].reshape(-1))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    losses.append(loss.item())
    
    # 计算准确率
    model.eval()
    with torch.no_grad():
        test_output = model(X_test, y_test, teacher_forcing_ratio=0.0)  # 推理时不用teacher forcing
        preds = test_output.argmax(dim=-1)
        mask = (y_test != 1)  # 非PAD位置
        acc = (preds == y_test)[mask].float().mean().item()
    accuracies.append(acc)
    
    if (epoch+1) % 20 == 0:
        print(f"Epoch {epoch+1}: Loss={loss.item():.4f}, Acc={acc:.2%}")

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(losses, 'b-', linewidth=1.5)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Seq2Seq训练损失 (序列反转任务)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(accuracies, 'g-', linewidth=1.5)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('测试准确率')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 二、注意力机制（Attention）

### Seq2Seq的瓶颈

标准Seq2Seq有一个严重缺陷：**编码器必须把所有信息压缩到一个固定长度的上下文向量中**。

输入序列越长，信息丢失越严重。这就像让你看完一本小说后只用一句话总结全部内容——必然有信息损失。

### 注意力机制的思想

**与其把所有信息压缩到一个向量，不如让解码器在每一步都能看到所有编码器的输出，并自己决定"关注"哪些部分。**

### 类比

翻译时，你不会记住整个句子的全部细节才开始翻译。你会：
1. 看源句子的不同部分
2. 对每个要翻译的单词，回头看源句子中相关的部分
3. 根据相关程度分配"注意力权重"

### 注意力计算步骤

1. **计算注意力分数**：decoder的隐藏状态 vs 所有encoder的隐藏状态
   $$e_{t,i} = \text{score}(h_t^{dec}, h_i^{enc})$$

2. **归一化为权重**：
   $$\alpha_{t,i} = \frac{\exp(e_{t,i})}{\sum_j \exp(e_{t,j})}$$

3. **加权求和编码器输出**：
   $$c_t = \sum_i \alpha_{t,i} h_i^{enc}$$

4. **用上下文向量辅助解码**

In [ ]:
# 注意力机制可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 图1: 无注意力的Seq2Seq
ax = axes[0]
ax.set_title('无注意力的Seq2Seq\n(信息瓶颈)', fontsize=12)

encoder_words = ['Je', 'suis', 'étudiant']
decoder_words = ['I', 'am', 'a', 'student']

# Encoder
for i, word in enumerate(encoder_words):
    ax.text(i*1.5, 3, word, ha='center', va='center', fontsize=12,
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))
    if i > 0:
        ax.annotate('', xy=(i*1.5-0.2, 3), xytext=((i-1)*1.5+0.2, 3),
                   arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

# 上下文向量
ax.text(2.25, 1.5, 'Context Vector\n(压缩所有信息)', ha='center', va='center', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7, edgecolor='red', lw=2))
ax.annotate('', xy=(2.25, 2.3), xytext=(2.25, 2.7),
           arrowprops=dict(arrowstyle='->', color='red', lw=2))

# Decoder
for i, word in enumerate(decoder_words):
    ax.text(i*1.5-0.75, 0, f'<{word}>', ha='center', va='center', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.7))
    if i > 0:
        ax.annotate('', xy=(i*1.5-0.95, 0), xytext=((i-1)*1.5-0.55, 0),
                   arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

ax.annotate('', xy=(2.25, 0.4), xytext=(2.25, 1.0),
           arrowprops=dict(arrowstyle='->', color='red', lw=2))

ax.text(2.25, -1.2, '问题: 长句子的信息会在压缩中丢失', ha='center',
        fontsize=10, color='red', style='italic')

ax.set_xlim(-2, 6)
ax.set_ylim(-1.5, 4)
ax.axis('off')

# 图2: 带注意力的Seq2Seq
ax = axes[1]
ax.set_title('带注意力的Seq2Seq\n(每一步关注不同部分)', fontsize=12)

# Encoder
for i, word in enumerate(encoder_words):
    ax.text(i*2, 3, word, ha='center', va='center', fontsize=12,
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))
    if i > 0:
        ax.annotate('', xy=(i*2-0.2, 3), xytext=((i-1)*2+0.2, 3),
                   arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

# Decoder
for i, word in enumerate(decoder_words):
    ax.text(i*1.8-0.9, 0, f'<{word}>', ha='center', va='center', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.7))

# 注意力连接线
attention_patterns = [
    # (decoder_word_idx, [(encoder_word_idx, weight), ...])
    (0, [(0, 0.7), (1, 0.2), (2, 0.1)]),   # "I" 主要关注 "Je"
    (1, [(0, 0.1), (1, 0.8), (2, 0.1)]),   # "am" 主要关注 "suis"
    (2, [(0, 0.05), (1, 0.05), (2, 0.9)]), # "a" 主要关注 "étudiant"
    (3, [(0, 0.05), (1, 0.1), (2, 0.85)]), # "student" 主要关注 "étudiant"
]

colors_attention = ['blue', 'green', 'red', 'purple']

for dec_idx, (d_idx, attn_weights) in enumerate(attention_patterns):
    for enc_idx, weight in attn_weights:
        ax.annotate('',
                   xy=(enc_idx*2, 2.6),
                   xytext=(d_idx*1.8-0.9, 0.4),
                   arrowprops=dict(arrowstyle='->', color=colors_attention[d_idx],
                                 alpha=weight*0.8, lw=weight*4))

ax.text(2.5, -1.2, '优势: 每一步都可以看到完整的输入序列', ha='center',
        fontsize=10, color='green', style='italic')

ax.set_xlim(-2, 6)
ax.set_ylim(-1.5, 4)
ax.axis('off')

plt.tight_layout()
plt.show()

## 三、注意力分数的计算方式

注意力的核心是如何计算decoder状态和encoder状态之间的"相关性"（注意力分数）。

### 三种主流方法

**1. Additive (Bahdanau) Attention：**
$$e_{t,i} = v^T \tanh(W h_t^{dec} + U h_i^{enc})$$
- 用一个前馈网络计算注意力
- 表达能力强，但计算量大

**2. Dot-Product (Luong) Attention：**
$$e_{t,i} = (h_t^{dec})^T h_i^{enc}$$
- 简单的点积
- 快速，但要求 $h^{dec}$ 和 $h^{enc}$ 维度相同

**3. Scaled Dot-Product（Transformer使用）：**
$$e_{t,i} = \frac{(h_t^{dec})^T h_i^{enc}}{\sqrt{d_k}}$$
- 除以 $\sqrt{d_k}$ 防止点积过大导致softmax梯度消失
- Transformer的核心机制

In [ ]:
# 三种注意力计算方式的对比
print("=" * 70)
print("三种注意力分数计算方式对比")
print("=" * 70)
print()

torch.manual_seed(42)

# 示例数据
d_k = 64  # 键向量维度
seq_len_enc = 5  # 编码器序列长度
decoder_hidden = torch.randn(d_k)  # 解码器隐藏状态
encoder_outputs = torch.randn(seq_len_enc, d_k)  # 编码器输出

print(f"解码器隐藏状态 shape: {decoder_hidden.shape}")
print(f"编码器输出 shape: {encoder_outputs.shape}")
print()

# 1. Additive Attention
W = nn.Linear(d_k, d_k)
U = nn.Linear(d_k, d_k)
v = nn.Linear(d_k, 1, bias=False)

scores_additive = v(torch.tanh(W(decoder_hidden) + U(encoder_outputs))).squeeze(-1)
weights_additive = F.softmax(scores_additive, dim=-1)

print(f"1. Additive (Bahdanau) Attention:")
print(f"   分数: {scores_additive.detach().numpy()}")
print(f"   权重: {weights_additive.detach().numpy()}")
print()

# 2. Dot-Product Attention
scores_dot = encoder_outputs @ decoder_hidden
weights_dot = F.softmax(scores_dot, dim=-1)

print(f"2. Dot-Product (Luong) Attention:")
print(f"   分数: {scores_dot.detach().numpy()}")
print(f"   权重: {weights_dot.detach().numpy()}")
print()

# 3. Scaled Dot-Product Attention
scores_scaled = (encoder_outputs @ decoder_hidden) / math.sqrt(d_k)
weights_scaled = F.softmax(scores_scaled, dim=-1)

print(f"3. Scaled Dot-Product Attention (Transformer):")
print(f"   缩放因子: 1/√{d_k} = {1/math.sqrt(d_k):.4f}")
print(f"   分数: {scores_scaled.detach().numpy()}")
print(f"   权重: {weights_scaled.detach().numpy()}")
print()

# 为什么需要缩放？
print("为什么需要缩放因子 1/√d_k ？")
print("-" * 50)

for d in [4, 16, 64, 256, 1024]:
    # 假设 q 和 k 的分量独立采样自均值为0方差为1的正态分布
    # 则点积 q·k 的方差为 d
    print(f"  d_k={d:4d}: 点积的标准差 ≈ √{d} = {math.sqrt(d):.1f}")

print("\n当 d_k 很大时，点积值会非常大，softmax 会进入梯度趋近于0的区域。")
print("除以 √d_k 可以将方差归一化为1，保持softmax在梯度敏感的区域。")

In [ ]:
# 可视化注意力权重
print("\n=== 注意力权重可视化 ===")
print()

# 模拟一个翻译任务的注意力权重
src_words = ['Je', 'suis', 'étudiant', 'en', 'université']
tgt_words = ['I', 'am', 'a', 'student', 'at', 'university']

# 模拟的注意力矩阵 (tgt_len x src_len)
attn_matrix = np.array([
    [0.85, 0.05, 0.03, 0.04, 0.03],  # "I" → "Je"
    [0.05, 0.80, 0.05, 0.05, 0.05],  # "am" → "suis"
    [0.03, 0.05, 0.80, 0.07, 0.05],  # "a" → "étudiant"
    [0.02, 0.03, 0.70, 0.10, 0.15],  # "student" → "étudiant"
    [0.03, 0.05, 0.10, 0.72, 0.10],  # "at" → "en"
    [0.02, 0.03, 0.15, 0.05, 0.75],  # "university" → "université"
])

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(attn_matrix, cmap='Blues', interpolation='nearest')

ax.set_xticks(range(len(src_words)))
ax.set_xticklabels(src_words, fontsize=12)
ax.set_yticks(range(len(tgt_words)))
ax.set_yticklabels(tgt_words, fontsize=12)

ax.set_xlabel('输入 (法语)', fontsize=12)
ax.set_ylabel('输出 (英语)', fontsize=12)
ax.set_title('注意力权重热力图\n(每行和为1)', fontsize=14)

# 在单元格中标数值
for i in range(len(tgt_words)):
    for j in range(len(src_words)):
        text = ax.text(j, i, f'{attn_matrix[i,j]:.2f}',
                      ha="center", va="center", color="black" if attn_matrix[i,j] < 0.5 else "white",
                      fontsize=10)

fig.colorbar(im, ax=ax, label='注意力权重')
plt.tight_layout()
plt.show()

print("解读:")
print("  - 每行代表一个输出词对所有输入词的注意力分布")
print("  - 颜色越深表示注意力权重越大")
print("  - 对角线附近颜色深表示对齐关系清晰")
print("  - 'I' 85%的注意力集中在 'Je' 上，说明对齐很好")

## 四、多头注意力（Multi-Head Attention）

### 为什么需要多头？

单个注意力机制只能关注一种模式。但语言中一个词可能和多个词有不同关系：

"The cat sat on the mat because it was tired."
- "it" 指代 "cat"（语法关系：指代消解）
- "tired" 描述 "cat" 的状态（语义关系）
- "sat" 和 "on the mat" 有位置关系（空间关系）

多头注意力允许模型**同时从不同角度关注信息**。

### 计算过程

1. 将 $Q, K, V$ 分成 $h$ 个头
2. 每个头独立计算注意力
3. 拼接所有头的结果
4. 通过线性变换融合

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h) W^O$$
$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

In [ ]:
# 多头注意力演示
print("=" * 70)
print("多头注意力机制详解")
print("=" * 70)
print()

torch.manual_seed(42)

# 参数
d_model = 64    # 总维度
n_heads = 4     # 头数
d_k = d_model // n_heads  # 每头维度 = 16
seq_len = 8

print(f"配置: d_model={d_model}, n_heads={n_heads}, d_k={d_k}")
print(f"每个头的维度: {d_k} (总维度 {d_model} / 头数 {n_heads})")
print()

# 创建Q, K, V
Q = torch.randn(seq_len, d_model)
K = torch.randn(seq_len, d_model)
V = torch.randn(seq_len, d_model)

# 每个头的线性投影
W_Q = nn.Linear(d_model, d_model)
W_K = nn.Linear(d_model, d_model)
W_V = nn.Linear(d_model, d_model)
W_O = nn.Linear(d_model, d_model)

# 多头注意力计算
Q_proj = W_Q(Q).view(seq_len, n_heads, d_k).transpose(0, 1)  # (n_heads, seq_len, d_k)
K_proj = W_K(K).view(seq_len, n_heads, d_k).transpose(0, 1)
V_proj = W_V(V).view(seq_len, n_heads, d_k).transpose(0, 1)

# 每个头独立计算注意力
print("各头注意力计算:")
attention_outputs = []
for head in range(n_heads):
    q = Q_proj[head]  # (seq_len, d_k)
    k = K_proj[head]
    v = V_proj[head]
    
    # Scaled dot-product attention
    scores = q @ k.T / math.sqrt(d_k)
    attn_weights = F.softmax(scores, dim=-1)
    head_output = attn_weights @ v  # (seq_len, d_k)
    
    attention_outputs.append(head_output)
    
    # 打印第一个查询词的注意力权重
    print(f"  Head {head+1}: 查询词0的注意力分布 = {attn_weights[0].detach().numpy()}")

print()

# 拼接+输出变换
concat_output = torch.cat(attention_outputs, dim=-1)  # (seq_len, d_model)
final_output = W_O(concat_output)

print(f"拼接后形状: {concat_output.shape}")
print(f"最终输出形状: {final_output.shape}")

# PyTorch内置多头注意力验证
mha = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
output_torch, attn_weights_torch = mha(Q.unsqueeze(0), K.unsqueeze(0), V.unsqueeze(0))
print(f"\nPyTorch MultiheadAttention输出形状: {output_torch.shape}")
print(f"注意力权重形状: {attn_weights_torch.shape}")

# 可视化各头的注意力模式
fig, axes = plt.subplots(1, n_heads, figsize=(16, 3))
for head in range(n_heads):
    q = Q_proj[head]
    k = K_proj[head]
    scores = q @ k.T / math.sqrt(d_k)
    attn_weights = F.softmax(scores, dim=-1)
    
    ax = axes[head]
    im = ax.imshow(attn_weights.detach().numpy(), cmap='viridis', interpolation='nearest')
    ax.set_title(f'Head {head+1}')
    ax.set_xlabel('Key')
    if head == 0:
        ax.set_ylabel('Query')

plt.suptitle('Multi-Head Attention: 各头的注意力权重', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 五、本章总结

### 核心要点

1. **Seq2Seq (Encoder-Decoder)**
   - 编码器压缩输入为上下文向量
   - 解码器从上下文向量逐步生成输出
   - 瓶颈：固定长度向量难以表示任意长度序列

2. **注意力机制**
   - 让解码器在每一步都能看到所有编码器输出
   - 通过注意力权重动态决定关注哪些部分
   - Scaled Dot-Product: $\text{softmax}(\frac{QK^T}{\sqrt{d_k}})V$

3. **多头注意力**
   - 同时从多个角度（子空间）计算注意力
   - 每个头学习不同的关系模式
   - 拼接后用线性变换融合

### 从Attention到Transformer

Transformer的核心创新：
- **完全抛弃RNN**，仅用自注意力机制
- **自注意力**：序列中的每个token都关注序列中的所有token
- **并行计算**：不需要像RNN那样按时间步顺序计算

### 下一篇预告

下一篇将深入**Transformer的完整架构**！